In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("desafio_07_joins")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 14:10:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/29 14:10:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
#Datasets menores
df_clientes = spark.read.csv("/workspace/datasets/07_joins/clientes.csv", header= True, inferSchema= True)
df_pedidos = spark.read.csv("/workspace/datasets/07_joins/pedidos.csv", header= True, inferSchema= True)
df_pagamentos = spark.read.csv("/workspace/datasets/07_joins/pagamentos.csv", header= True, inferSchema= True)

In [4]:
df_pagamentos.show()

+------------+---------+---------------+----------------+----------+--------------+
|pagamento_id|pedido_id|forma_pagamento|status_pagamento|valor_pago|data_pagamento|
+------------+---------+---------------+----------------+----------+--------------+
|       90001|    10001|  Cartão Débito|        aprovado|    473.56|    2025-10-12|
|       90002|    10002|            Pix|        aprovado|    303.13|    2025-08-03|
|       90003|    10003|            Pix|        aprovado|    281.64|    2025-10-08|
|       90004|    10004| Cartão Crédito|        aprovado|    281.65|    2025-09-04|
|       90005|    10005|            Pix|        aprovado|    893.22|    2025-09-18|
|       90006|    10007|         Boleto|        aprovado|    233.26|    2025-10-02|
|       90007|    10008| Cartão Crédito|        aprovado|    487.87|    2025-05-07|
|       90008|    10009| Cartão Crédito|        aprovado|    399.09|    2025-02-01|
|       90009|    10010|            Pix|        aprovado|     50.94|    2025

Parte A — Validar cardinalidade

In [5]:
df_validacao_clientes = (
    df_clientes.groupBy("cliente_id")
    .agg(F.count("cliente_id").alias("quantidade_clientes"))
    .filter(F.col("quantidade_clientes") > 1)
)

In [6]:
df_validacao_clientes.show()

+----------+-------------------+
|cliente_id|quantidade_clientes|
+----------+-------------------+
+----------+-------------------+



In [7]:
df_pedidos_existentes = (
    df_pedidos.groupBy("cliente_id")
    .agg(F.count("*").alias("quantidade_pedidos"))
    
)

In [8]:
df_pedidos_existentes.show()

+----------+------------------+
|cliente_id|quantidade_pedidos|
+----------+------------------+
|       463|                 7|
|       148|                13|
|       243|                19|
|       392|                 8|
|       516|                12|
|        31|                10|
|        85|                11|
|       137|                 9|
|       251|                 8|
|       451|                 7|
|       458|                11|
|        65|                 6|
|        53|                14|
|       255|                10|
|       133|                13|
|       296|                10|
|        78|                16|
|       513|                 6|
|       322|                10|
|       321|                10|
+----------+------------------+
only showing top 20 rows



Explicação sobre 

- 1:1
- 1:N
- N:N

Resposta: A partir dessas duas validações é possivel perceber que, apenas um cliente pode ter varios pedidos demonstrando o comportamento de 1:N (um para muitos). Sobre a primeira, percebe-se que, cada cliente so pode ser referenciado por um unico id, tornando esse comportamento 1:1 (um para um)

Parte B — Clientes com pedidos

In [9]:
df_clientes_com_pedidos = df_clientes.join(
    df_pedidos,
    df_clientes.cliente_id == df_pedidos.cliente_id,
    "left_semi"
)

In [10]:
df_clientes_sem_pedidos = df_clientes.join(
    df_pedidos,
    df_clientes.cliente_id == df_pedidos.cliente_id,
    "left_anti"
)

In [11]:
#df_clientes_com_pedidos.show()
#df_clientes_sem_pedidos.show()

Parte C — Qualidade de dados


In [12]:
df_pedidos_orfaos = df_pedidos.join(
    df_clientes,
    df_pedidos.cliente_id == df_clientes.cliente_id,
    "left_anti"
)

In [13]:
df_pagamentos_orfaos = df_pagamentos.join(
    df_pedidos,
    df_pagamentos.pedido_id == df_pedidos.pedido_id,
    "left_anti"
)

Explique por que a tabela que você deseja retornar precisa ficar à esquerda.

Resposta: Ela precisa ficar a esquerda pois nesse cenario precisamos retornar precisamente todos os dados que não fazem match com a tabela da direita, caso haja dados que deem match com ela, os mesmo não serão retornados

Parte D — Pipeline com três tabelas

In [14]:
clientes = df_clientes.alias("cli")
pedidos = df_pedidos.alias("ped")
pagamentos = df_pagamentos.alias("pag")

In [15]:
df_clientes_pedidos = clientes.join(
    pedidos,
    clientes.cliente_id == pedidos.cliente_id,
    "inner"
)

In [16]:
df_clientes_pedidos_pagamentos = df_clientes_pedidos.join(
    pagamentos,
    df_clientes_pedidos.pedido_id == pagamentos.pedido_id,
    "left"
)

In [17]:
df_pipeline_final = (
    df_clientes_pedidos_pagamentos.select(
        F.col("cli.cliente_id"),
        F.col("cli.estado"),
        F.col("ped.pedido_id"),
        F.col("ped.valor_total"),
        F.col("pag.pagamento_id"),
        F.col("pag.forma_pagamento"),
        F.col("pag.valor_pago")
    )
)

In [18]:
df_pipeline_final_nulos = (
    df_pipeline_final.filter(F.col("pag.pagamento_id").isNull())
)

In [19]:
df_pipeline_final_nulos.show()

+----------+------+---------+-----------+------------+---------------+----------+
|cliente_id|estado|pedido_id|valor_total|pagamento_id|forma_pagamento|valor_pago|
+----------+------+---------+-----------+------------+---------------+----------+
|       164|    PR|    10006|     562.28|        NULL|           NULL|      NULL|
|       370|    RJ|    10040|     292.38|        NULL|           NULL|      NULL|
|        60|    MG|    10045|     327.27|        NULL|           NULL|      NULL|
|       414|    SP|    10050|      431.7|        NULL|           NULL|      NULL|
|       352|    MG|    10052|     227.09|        NULL|           NULL|      NULL|
|       388|    PR|    10055|     360.44|        NULL|           NULL|      NULL|
|       117|    SP|    10068|     329.52|        NULL|           NULL|      NULL|
|       193|    RJ|    10084|     262.86|        NULL|           NULL|      NULL|
|       133|    MG|    10092|     414.46|        NULL|           NULL|      NULL|
|       100|    

Explique por que valores nulos permaneceram no resultado.

Resposta: Valores nulos apareceram no resultado pois como fizemos um left join no resultado do primeiro join, o spark deu proridade ao lado esquerdo do join trazendo todos os dados da tabela de pedidos, mesmo aqueles que não dão match com pedido_id

Parte E — Verificação de multiplicação de linhas

In [20]:
df_clientes_pedidos.count()

4834

In [21]:
df_pipeline_final.count()

4834

In [22]:
df_pagamentos_verificacao = (
    df_pagamentos.groupBy("pedido_id")
    .agg(F.count("*").alias("quantidade_pagamentos"))
    .filter(F.col("quantidade_pagamentos") > 1)
)

In [23]:
df_pagamentos_verificacao.show()

+---------+---------------------+
|pedido_id|quantidade_pagamentos|
+---------+---------------------+
|    20082|                    2|
|    20001|                    2|
+---------+---------------------+



Por que uma chave duplicada no lado direito pode multiplicar linhas de um join?

Resposta: Uma chave duplicada no lado direito multiplica as linhas de um join porque o spark liga cada correspondência encontrada. Se uma linha da tabela esquerda encontra múltiplos registros iguais na tabela direita, ela se repete para cada um deles.

Por que uma relação N:N pode inflar métricas como SUM(valor_total)?

Resposta: Uma relação N:N (Muitos para Muitos) infla métricas como SUM(valor_total) porque ela gera um efeito multiplicador no cruzamento dos dados, fazendo com que o mesmo valor seja somado várias vezes no resultado final.

Parte F — Join com chave composta

In [24]:
dados_1 = [(10, "BA"), (10, "SP")]
colunas1 = ["cliente_id", "estado"]

In [25]:
dados_2 = [(10, "BA", "X"), (10, "SP", "Y")]
colunas_2 = ["cliente_id", "estado", "codigo"]

In [26]:
df_1 = spark.createDataFrame(dados_1, schema= colunas1)
df_2 = spark.createDataFrame(dados_2, schema= colunas_2)

In [27]:
df1 = df_1.alias("df1")
df2 = df_2.alias("df2")

In [28]:
df_join_clientes = df1.join(
    df2,
    df1.cliente_id == df2.cliente_id,
    "left"
)

In [29]:
df_join_clientes.show()

+----------+------+----------+------+------+
|cliente_id|estado|cliente_id|estado|codigo|
+----------+------+----------+------+------+
|        10|    BA|        10|    SP|     Y|
|        10|    BA|        10|    BA|     X|
|        10|    SP|        10|    SP|     Y|
|        10|    SP|        10|    BA|     X|
+----------+------+----------+------+------+



In [30]:
df_join_clientes_estado = df1.join(
    df2,
    (F.col("df1.cliente_id") == F.col("df2.cliente_id")) &
    (F.col("df1.estado") == F.col("df2.estado")),
    "left"
)

In [31]:
df_join_clientes_estado.show()

+----------+------+----------+------+------+
|cliente_id|estado|cliente_id|estado|codigo|
+----------+------+----------+------+------+
|        10|    BA|        10|    BA|     X|
|        10|    SP|        10|    SP|     Y|
+----------+------+----------+------+------+



Compare a quantidade de linhas e explique por que uma chave incompleta pode criar correspondências indevidas.

Resposta: Fazendo join apenas com cliente_id o spark multiplicou as linhas resultando em 4, ja com chave dupla, mantiveram-se 2 linhas. Uma chave incompleta pode criar correspondencias indevidas pois o spark pode encontrar os mesmo registros do dataframe a esquerda, com o dataframe da direita, com a chave composta o resultado teve 2 linhas porque ambas as informações precisavam ser iguais.

Parte G — Physical Plan e performance

In [32]:
df_clientes_large = spark.read.csv("/workspace/datasets/07_joins/large_csvs/clientes_large.csv", header= True, inferSchema= True)
df_pedidos_large = spark.read.csv("/workspace/datasets/07_joins/large_csvs/pedidos_large.csv", header= True, inferSchema= True)

In [33]:
df_clientes_pedidos_large = df_clientes_large.join(
    df_pedidos_large,
    df_pedidos_large.cliente_id == df_clientes_large.cliente_id,
    "inner"
)

In [34]:
df_clientes_pedidos_large.explain(True)

== Parsed Logical Plan ==
Join Inner, (cliente_id#508 = cliente_id#478)
:- Relation [cliente_id#478,cidade#479,estado#480,segmento#481,data_cadastro#482,ativo#483] csv
+- Relation [pedido_id#507,cliente_id#508,data_pedido#509,status#510,canal#511,valor_total#512] csv

== Analyzed Logical Plan ==
cliente_id: int, cidade: string, estado: string, segmento: string, data_cadastro: date, ativo: boolean, pedido_id: int, cliente_id: int, data_pedido: date, status: string, canal: string, valor_total: double
Join Inner, (cliente_id#508 = cliente_id#478)
:- Relation [cliente_id#478,cidade#479,estado#480,segmento#481,data_cadastro#482,ativo#483] csv
+- Relation [pedido_id#507,cliente_id#508,data_pedido#509,status#510,canal#511,valor_total#512] csv

== Optimized Logical Plan ==
Join Inner, (cliente_id#508 = cliente_id#478)
:- Filter isnotnull(cliente_id#478)
:  +- Relation [cliente_id#478,cidade#479,estado#480,segmento#481,data_cadastro#482,ativo#483] csv
+- Filter isnotnull(cliente_id#508)
   +- R

Indentifique explique: 

- Exchange
- Shuffle
- Sort
- SortMergeJoin

E por que existem dois Exchange hashpartitioning(cliente_id, ...)

Resposta: Como temos datasets gigantesco o spark realiza os seguintes metodos, no pshycal podemos indentificar o filescan csv, ele representa leitura dos dados e  indentificou que trata-se de arquivo csv em seguida acontece o shuffle, ou seja, nesse momento o spark redistribui os dados pela chave entre partições, dentro de cada partição ocorre o metodo sort, nesse momento os dados são colocados em ordem crescente em cada partição para faciltar a proxima etapa que é o sortmergejoin, essa é a etapa final onde os dados são finalmente juntados. Sobre a existencia de dois Exchange hashpartitioning(cliente_id, ...), ocorre por conta dos dois dataframes que estão sendo utilizados, então para cada tabela temos um Exchange hashpartitioning

In [35]:
df_pedidos_large_clientes_small = df_pedidos_large.join(
    F.broadcast(df_clientes),
    df_pedidos_large.cliente_id == df_clientes.cliente_id,
    "inner"
)

Comparação entre

- BroadcastHashJoin
- SortMergeJoin

Resposta: Ambos são planos de execução de joins realizados pelo spark, porém cada um com caracteristicas diferentes, no BroadcastHashJoin geralmente são tabelas menores, com menores quantidades de linhas e com poucos bytes, nesse processo o spark replica o lado pequeno, evita grande shuffle bilateral, o spark tambem utiliza buildleft/buildght para indicar o lado do hash, nesse join acima utilizei a tabela de clientes como tabela da esquerda entao o spark utilizou o BuildLeft. Sobre o SortMergeJoin, ele ocorre quando spark indentifica que os dois dataframes são gigantesco tanto em linha quanto em bytes, ele pode ate ser reduzido em quantidade de linhas, mas se as informações das colunas forem muito grande, e isso acabar pesando em bytes o spark utilizara o SortMergeJoin nesse processo o spark redistribui os dados pela chave, ocorre o shuffle bilateral, os dados são redistribuidos entre partições, apos isso vem o sort onde dentro dessas partições o spark coloca os dados em ordem crescente, e por fim ocorre o merge final das chaves correspondentes.

Questão final da aula

Em uma célula Markdown, responda com suas palavras:
Você possui uma tabela de fatos com 1 bilhão de registros e uma dimensão com poucos MB. Qual estratégia de join você esperaria que fosse mais adequada e por quê?

Resposta: Nesse cenario a estrategia de broadcast utilizando a tabela dimensão seria mais inteligente, aqui evitariamos um shuffle bilateral 

E depois:
Se os dois lados tiverem centenas de milhões de registros e nenhum for adequado para broadcast, o que você espera encontrar no Physical Plan?

Resposta: Nesse cenario o sortmerge entraria em cena, redistribuindo os dados entre partições pela chave, ordenando de forma crescente os dados dentro de cada partição e por fim fazendo o join final

In [36]:
spark.stop()